# SyriaTel Customer Churn Prediction - Phase 3 project

**Author:** Angela Mukami K

**Date:** February 2026

---

## Table of Contents

1. [Business Understanding](#business-understanding)
2. [Data Understanding](#data-understanding)
3. [Data Preparation](#data-preparation)
4. [Modeling](#modeling)
5. [Evaluation](#evaluation)
6. [Conclusions and Recommendations](#conclusions-and-recommendations)

---

## 1. Business Understanding <a id='business-understanding'></a>

### Stakeholder
SyriaTel, a telecommunications company

### Business Problem
SyriaTel is experiencing customer churn, which directly impacts revenue and profitability. When customers leave, the company loses not only their recurring revenue but also the investment made in acquiring them. 

### Project Objective
Build a classification model to predict whether a customer will churn (stop doing business with SyriaTel) based on their account information and usage patterns.

### Success Criteria
- Identify key features that predict customer churn
- Build a model with strong predictive performance
- Provide actionable recommendations to reduce churn

### Why Classification?
This is a binary classification problem because our target variable (churn) is categorical:
- Class 0: Customer stays
- Class 1: Customer churns

We need to predict which category a customer falls into, not a numeric value.

---

## 2. Data Understanding <a id='data-understanding'></a>

### 2.1 Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Machine Learning - Evaluation
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix, 
    classification_report,
    roc_auc_score,
    roc_curve
)

# Model tuning
from sklearn.model_selection import GridSearchCV, cross_val_score

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully!")

### 2.2 Load Data

In [ ]:
# Load the dataset
# Update the filename below to match your dataset file name
df = pd.read_csv('data/bigml_59c28831336c6604c800002a.csv')  # Update this path

# Display first few rows
df.head()

### 2.3 Initial Data Exploration

In [ ]:
# Dataset shape
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

### 2.4 Target Variable Analysis

In [ ]:
# Assuming 'churn' is the target variable - update if different
target_col = 'churn'  # Update this if your target column has a different name

# Check target variable distribution
print("Target variable distribution:")
print(df[target_col].value_counts())
print("\nPercentages:")
print(df[target_col].value_counts(normalize=True) * 100)

In [ ]:
# Visualize target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Customer Churn Distribution')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Pie chart
df[target_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[1].set_title('Customer Churn Percentage')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

**Observations:**
- [Note if the dataset is balanced or imbalanced]
- [Any initial insights about the target variable]

### 2.5 Feature Analysis

In [ ]:
# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()

# Remove target from feature lists if present
if target_col in numeric_cols:
    numeric_cols.remove(target_col)
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Distribution of numeric features
df[numeric_cols].hist(figsize=(15, 12), bins=30, edgecolor='black')
plt.suptitle('Distribution of Numeric Features', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze categorical features
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print("-" * 50)

### 2.6 Correlation Analysis

In [ ]:
# Correlation matrix for numeric features
plt.figure(figsize=(12, 10))
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target variable (if numeric)
# Convert target to numeric if it's boolean/categorical
df_temp = df.copy()
if df_temp[target_col].dtype == 'object' or df_temp[target_col].dtype == 'bool':
    df_temp[target_col] = df_temp[target_col].astype(int)

correlations_with_target = df_temp[numeric_cols + [target_col]].corr()[target_col].sort_values(ascending=False)
print("Correlation with target variable:")
print(correlations_with_target)

In [ ]:
# Visualize top correlations with target
top_n = 10
top_correlations = correlations_with_target[1:top_n+1]  # Exclude target itself

plt.figure(figsize=(10, 6))
top_correlations.plot(kind='barh', color='teal')
plt.title(f'Top {top_n} Features Correlated with Churn')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

### 2.7 Relationship Between Features and Target

In [ ]:
# Analyze categorical features vs target
for col in categorical_cols[:5]:  # Limit to first 5 for brevity
    plt.figure(figsize=(10, 4))
    
    # Create crosstab
    ct = pd.crosstab(df[col], df[target_col], normalize='index') * 100
    
    ct.plot(kind='bar', stacked=False)
    plt.title(f'Churn Rate by {col}')
    plt.ylabel('Percentage')
    plt.xlabel(col)
    plt.legend(title='Churn')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plots for numeric features vs target
# Select a few important numeric features to visualize
important_features = numeric_cols[:6]  # Adjust based on your data

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(important_features):
    df.boxplot(column=col, by=target_col, ax=axes[idx])
    axes[idx].set_title(f'{col} by Churn')
    axes[idx].set_xlabel('Churn')

plt.tight_layout()
plt.show()

### 2.8 Key Insights from EDA

**Summary of findings:**
- [List key patterns observed]
- [Note any data quality issues]
- [Identify potential important features]
- [Class imbalance observations]
- [Any outliers or unusual patterns]

---

## 3. Data Preparation <a id='data-preparation'></a>

### 3.1 Handle Missing Values

In [ ]:
# Check again for missing values
print("Missing values:")
print(df.isnull().sum())

# If there are missing values, decide on a strategy:
# - Drop rows with missing values
# - Fill with mean/median/mode
# - Use more sophisticated imputation

# Example: Drop rows with missing values (if any)
# df = df.dropna()

# Example: Fill numeric columns with median
# for col in numeric_cols:
#     if df[col].isnull().sum() > 0:
#         df[col].fillna(df[col].median(), inplace=True)

print(f"\nDataset shape after handling missing values: {df.shape}")

### 3.2 Handle Duplicates

In [ ]:
# Remove duplicates if any
df = df.drop_duplicates()
print(f"Dataset shape after removing duplicates: {df.shape}")

### 3.3 Feature Engineering (Optional)

In [ ]:
# Create new features if needed
# Examples:
# df['total_minutes'] = df['total_day_minutes'] + df['total_eve_minutes'] + df['total_night_minutes']
# df['total_calls'] = df['total_day_calls'] + df['total_eve_calls'] + df['total_night_calls']
# df['average_call_duration'] = df['total_minutes'] / df['total_calls']

print("Feature engineering complete (if any created)")

### 3.4 Prepare Features and Target

In [ ]:
# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Convert target to binary if it's not already (e.g., True/False to 1/0)
if y.dtype == 'object' or y.dtype == 'bool':
    y = y.astype(int)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:\n{y.value_counts()}")

### 3.5 Handle Categorical Variables

In [ ]:
# Identify columns to drop (if any)
# Examples: phone number, state codes that won't be useful
columns_to_drop = []  # Add column names to drop, e.g., ['phone number']

if columns_to_drop:
    X = X.drop(columns=columns_to_drop)
    print(f"Dropped columns: {columns_to_drop}")

# Update categorical and numeric columns
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nCategorical features: {categorical_features}")
print(f"Numeric features: {numeric_features}")

In [ ]:
# Encode categorical variables
# For binary categories, use Label Encoding
# For multi-class categories, use One-Hot Encoding

from sklearn.preprocessing import LabelEncoder

# Example: if you have binary categorical columns
# for col in categorical_features:
#     if X[col].nunique() == 2:
#         le = LabelEncoder()
#         X[col] = le.fit_transform(X[col])

# For this dataset, we'll use pd.get_dummies for simplicity
if categorical_features:
    X = pd.get_dummies(X, columns=categorical_features, drop_first=True)
    print(f"\nShape after encoding: {X.shape}")
    print(f"New columns: {X.columns.tolist()}")

### 3.6 Train-Test Split

**Important:** We perform the train-test split BEFORE any transformations (scaling, etc.) to prevent data leakage.

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80-20 split
    random_state=42,    # For reproducibility
    stratify=y          # Maintain class distribution
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining set target distribution:\n{y_train.value_counts()}")
print(f"\nTest set target distribution:\n{y_test.value_counts()}")

### 3.7 Feature Scaling

**Critical:** We fit the scaler ONLY on training data and then transform both training and test data. This prevents data leakage.

In [ ]:
# Initialize the scaler
scaler = StandardScaler()

# Fit on training data and transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier manipulation (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Feature scaling complete!")
print(f"Scaled training data shape: {X_train_scaled.shape}")
print(f"Scaled test data shape: {X_test_scaled.shape}")

---

## 4. Modeling <a id='modeling'></a>

We will follow an iterative approach:
1. Start with a simple baseline model
2. Evaluate performance
3. Try different models
4. Tune hyperparameters
5. Compare and select the best model

### 4.1 Helper Functions for Evaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name="Model"):
    """
    Evaluate a classification model and print metrics.
    """
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    
    # Print results
    print(f"\n{'='*60}")
    print(f"{model_name} Performance")
    print(f"{'='*60}")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy:     {test_accuracy:.4f}")
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1-Score:          {f1:.4f}")
    
    # Check for overfitting
    if train_accuracy - test_accuracy > 0.05:
        print("\n⚠️  Warning: Possible overfitting detected!")
    
    print(f"\n{'='*60}\n")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    # Classification Report
    print("Classification Report:")
    print(classification_report(y_test, y_test_pred))
    
    return {
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

### 4.2 Baseline Model: Logistic Regression

We'll start with a simple logistic regression model as our baseline.

In [ ]:
# Initialize and train baseline model
baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

# Evaluate
baseline_results = evaluate_model(
    baseline_model, 
    X_train_scaled, 
    X_test_scaled, 
    y_train, 
    y_test,
    model_name="Baseline Logistic Regression"
)

**Baseline Model Observations:**
- [Note the performance metrics]
- [Identify areas for improvement]
- [Discuss which metric is most important for this business problem]

### 4.3 Model 2: Decision Tree

In [ ]:
# Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_scaled, y_train)

# Evaluate
dt_results = evaluate_model(
    dt_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    model_name="Decision Tree"
)

**Decision Tree Observations:**
- [Compare with baseline]
- [Note any overfitting issues]
- [Discuss interpretability]

### 4.4 Model 3: Random Forest

In [ ]:
# Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_scaled, y_train)

# Evaluate
rf_results = evaluate_model(
    rf_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    model_name="Random Forest"
)

### 4.5 Hyperparameter Tuning

Let's tune the hyperparameters of our best performing model so far.

In [ ]:
# Example: Tune Random Forest hyperparameters
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize GridSearchCV
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',  # Choose appropriate metric
    n_jobs=-1,
    verbose=1
)

# Fit grid search
print("Starting hyperparameter tuning...")
rf_grid.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {rf_grid.best_params_}")
print(f"Best cross-validation score: {rf_grid.best_score_:.4f}")

In [ ]:
# Evaluate tuned model
tuned_rf_results = evaluate_model(
    rf_grid.best_estimator_,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    model_name="Tuned Random Forest"
)

**Hyperparameter Tuning Observations:**
- [Discuss improvements from tuning]
- [Explain the rationale for parameter choices]
- [Compare with untuned model]

### 4.6 Additional Models (Optional)

In [ ]:
# Try other models if time permits
# Example: Gradient Boosting
# gb_model = GradientBoostingClassifier(random_state=42)
# gb_model.fit(X_train_scaled, y_train)
# gb_results = evaluate_model(gb_model, X_train_scaled, X_test_scaled, y_train, y_test, "Gradient Boosting")

### 4.7 Model Comparison

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame({
    'Model': ['Baseline Logistic Regression', 'Decision Tree', 'Random Forest', 'Tuned Random Forest'],
    'Test Accuracy': [
        baseline_results['test_accuracy'],
        dt_results['test_accuracy'],
        rf_results['test_accuracy'],
        tuned_rf_results['test_accuracy']
    ],
    'Precision': [
        baseline_results['precision'],
        dt_results['precision'],
        rf_results['precision'],
        tuned_rf_results['precision']
    ],
    'Recall': [
        baseline_results['recall'],
        dt_results['recall'],
        rf_results['recall'],
        tuned_rf_results['recall']
    ],
    'F1-Score': [
        baseline_results['f1_score'],
        dt_results['f1_score'],
        rf_results['f1_score'],
        tuned_rf_results['f1_score']
    ]
})

print("\nModel Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Test Accuracy', 'Precision', 'Recall', 'F1-Score']
axes = axes.ravel()

for idx, metric in enumerate(metrics):
    results_df.plot(x='Model', y=metric, kind='bar', ax=axes[idx], legend=False, color='steelblue')
    axes[idx].set_title(f'{metric} Comparison')
    axes[idx].set_ylabel(metric)
    axes[idx].set_xlabel('')
    axes[idx].set_ylim([0, 1])
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

## 5. Evaluation <a id='evaluation'></a>

### 5.1 Select Final Model

In [ ]:
# Select your final model (update based on your results)
final_model = rf_grid.best_estimator_  # or whichever model performed best
final_model_name = "Tuned Random Forest"

print(f"Selected final model: {final_model_name}")

### 5.2 Feature Importance Analysis

In [ ]:
# Get feature importances (for tree-based models)
if hasattr(final_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Display top features
    print("Top 10 Most Important Features:")
    print(feature_importance.head(10))
    
    # Plot feature importances
    plt.figure(figsize=(10, 6))
    feature_importance.head(10).plot(x='feature', y='importance', kind='barh', color='teal')
    plt.title('Top 10 Feature Importances')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type.")

### 5.3 ROC Curve and AUC Score

In [ ]:
# Get probability predictions
y_pred_proba = final_model.predict_proba(X_test_scaled)[:, 1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"\nAUC Score: {roc_auc:.4f}")

### 5.4 Model Limitations and Considerations

**Limitations:**
- [Discuss where the model performs poorly]
- [Note any class imbalance issues]
- [Identify types of customers the model might misclassify]
- [Discuss potential real-world deployment challenges]

**Considerations for Production:**
- [Model maintenance and retraining needs]
- [Data quality requirements]
- [Ethical considerations]
- [Potential for model drift]

---

## 6. Conclusions and Recommendations <a id='conclusions-and-recommendations'></a>

### 6.1 Summary of Findings

**Model Performance:**
- [Summarize final model performance]
- [Highlight key metrics relevant to the business problem]

**Key Predictors of Churn:**
- [List top 3-5 most important features]
- [Explain what these mean in business terms]

**Model Insights:**
- [What patterns did the model identify?]
- [Which customers are most at risk?]
- [What behaviors or characteristics predict churn?]

### 6.2 Business Recommendations

Based on the model findings, here are actionable recommendations for SyriaTel:

1. **[Recommendation 1 based on important feature]**
   - What to do: [specific action]
   - Expected impact: [potential benefit]
   - Implementation: [how to do it]

2. **[Recommendation 2 based on important feature]**
   - What to do: [specific action]
   - Expected impact: [potential benefit]
   - Implementation: [how to do it]

3. **[Recommendation 3 based on model performance]**
   - What to do: [specific action]
   - Expected impact: [potential benefit]
   - Implementation: [how to do it]

### 6.3 Next Steps

**For Model Improvement:**
- [Suggest additional features to collect]
- [Propose advanced techniques to try]
- [Mention ensemble methods or other algorithms]

**For Business Implementation:**
- [Pilot program suggestions]
- [A/B testing recommendations]
- [Monitoring and evaluation plan]

**For Further Analysis:**
- [Additional questions to explore]
- [Deeper dives into specific customer segments]
- [Cost-benefit analysis of interventions]

---

## Final Notes

This analysis has demonstrated the application of machine learning classification techniques to predict customer churn for SyriaTel. The iterative modeling approach allowed us to build progressively better models, and the final model provides actionable insights that can help reduce customer churn and improve business outcomes.

**Key Takeaways:**
- [Main finding 1]
- [Main finding 2]
- [Main finding 3]

---

*End of Analysis*